# Módulo 9: dashboard diario

La página `9_Dashboard.py` es **descriptiva**: resultados de la última jornada, partidos de la próxima con probabilidades del Elo, posiciones por conferencia (con probabilidad de playoffs del simulador del Módulo 6) y las mejores actuaciones fantasy. Casi no tiene lógica nueva: junta lo ya construido, y esa lógica vive en `sportslab/analysis/daily.py` (con tests).

**Pregunta de este notebook:** como el dashboard muestra probabilidades del Elo cada día, ¿qué tan fiables son *en la práctica*, partido a partido, y qué se puede concluir con los ~100 partidos que lleva la temporada actual?

**Diseño (decidido antes de ver los datos):**
- Probabilidades previas al partido (sin mirar el resultado) del Elo con los parámetros del Módulo 1 (K=24, ventaja de local 40, regresión 0.7).
- Referencias: moneda (log loss 0.693) y "gana el local" con la tasa histórica.
- Métricas: log loss, Brier, acierto del favorito, con intervalo; y cuántos partidos hacen falta para que una cifra signifique algo.

**Mi hipótesis previa:** ninguna ("sin intuición, corre").

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sportslab.analysis.daily import surprises
from sportslab.data import NbaClient
from sportslab.data.games import load_games, season_labels
from sportslab.data.schedule import load_schedule, schedule_to_games
from sportslab.models.elo import EloParams, run_elo
from sportslab.models.metrics import accuracy, brier_score, log_loss

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
pd.options.display.float_format = "{:.3f}".format

TUNED = EloParams(k=24, home_advantage=40, carryover=0.7)
client = NbaClient()
history = load_games(client, season_labels(2015, 2025))
current = schedule_to_games(load_schedule(client, "2026-27"), "2026-27")
pre = run_elo(pd.concat([history, current], ignore_index=True), TUNED)
now = pre[(pre["season"] == "2026-27") & ~pre["neutral_site"]]
last = pre[(pre["season"] == "2025-26") & ~pre["neutral_site"]]
print(f"2026-27 hasta ahora: {len(now)} partidos (sin sede neutral); 2025-26 completa: {len(last)}")

2026-27 hasta ahora: 107 partidos (sin sede neutral); 2025-26 completa: 1225


## 1. ¿Cómo le va al Elo en lo que va de 2026-27?

In [2]:
def scores(df):
    y, p = df["home_win"].to_numpy(), df["p_home"].to_numpy()
    return {
        "partidos": len(df),
        "log loss": log_loss(y, p),
        "Brier": brier_score(y, p),
        "acierto favorito": accuracy(y, p),
    }


home_rate = history.loc[~history["neutral_site"], "home_win"].mean()
rows = {"Elo (2026-27)": scores(now), "Elo (2025-26, prueba del Módulo 1)": scores(last)}
for label, df in (("2026-27", now), ("2025-26", last)):
    ref = df.assign(p_home=0.5)
    rows[f"moneda ({label})"] = scores(ref)
    rows[f"'gana el local' ({label})"] = scores(df.assign(p_home=home_rate))
print(pd.DataFrame(rows).T.to_string())

# how uncertain is a log loss measured on n games? per-game bootstrap
rng = np.random.default_rng(0)
per_game = -(np.where(now["home_win"] == 1, np.log(now["p_home"]), np.log(1 - now["p_home"])))
boot = [rng.choice(per_game, len(per_game)).mean() for _ in range(2000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(
    f"\nLog loss 2026-27: {per_game.mean():.3f}, IC 95 % [{lo:.3f}, {hi:.3f}] con {len(now)} partidos"
)

                                    partidos  log loss  Brier  acierto favorito
Elo (2026-27)                        107.000     0.623  0.218             0.636
Elo (2025-26, prueba del Módulo 1)  1225.000     0.604  0.208             0.681
moneda (2026-27)                     107.000     0.693  0.250             0.500
'gana el local' (2026-27)            107.000     0.698  0.252             0.514
moneda (2025-26)                    1225.000     0.693  0.250             0.500
'gana el local' (2025-26)           1225.000     0.687  0.247             0.554

Log loss 2026-27: 0.623, IC 95 % [0.555, 0.691] con 107 partidos


**Lectura.** En lo que va de 2026-27 el Elo tiene log loss 0.623 (acierto del favorito 64 %), mejor que la moneda (0.693) y que "gana el local" (0.698). Pero con 107 partidos el intervalo del 95 % va de 0.555 a 0.691 (±0.07): incluye valores tan buenos como el de 2025-26 (0.604) y llega casi hasta la moneda. Una o dos semanas de dashboard no bastan para confiar o desconfiar del Elo: la evidencia sólida son los 1225 partidos de prueba de 2025-26 (Módulo 1). La página incluye esa advertencia junto a las probabilidades.

## 2. ¿Cuántas "sorpresas" son normales?

El dashboard marca como sorpresa un partido que ganó quien tenía ≤ 35 % de probabilidad. Si el Elo está bien calibrado, ¿qué fracción de los partidos debería ser sorpresa?

In [3]:
sur_last = surprises(last.assign(played=True), 0.35)
mask = np.minimum(last["p_home"], 1 - last["p_home"]) <= 0.35
exp_surprises = float(np.minimum(last.loc[mask, "p_home"], 1 - last.loc[mask, "p_home"]).sum())
print(
    f"2025-26: {int(mask.sum())} partidos con un lado ≤ 35 %; sorpresas observadas {len(sur_last)}; "
    f"esperadas si el Elo está calibrado: {exp_surprises:.0f}"
)
now_mask = np.minimum(now["p_home"], 1 - now["p_home"]) <= 0.35
sur_now = surprises(now.assign(played=True), 0.35)
exp_now = float(np.minimum(now.loc[now_mask, "p_home"], 1 - now.loc[now_mask, "p_home"]).sum())
print(
    f"2026-27: {int(now_mask.sum())} partidos con un lado ≤ 35 %; sorpresas observadas {len(sur_now)}; esperadas {exp_now:.1f}"
)

2025-26: 623 partidos con un lado ≤ 35 %; sorpresas observadas 149; esperadas si el Elo está calibrado: 155
2026-27: 54 partidos con un lado ≤ 35 %; sorpresas observadas 15; esperadas 13.8


En 2025-26 hubo 149 sorpresas de 623 partidos con un lado ≤ 35 %, frente a ~155 esperadas si el Elo estuviera calibrado: coincide. En lo que va de 2026-27, 15 observadas contra ~14 esperadas. Con tan pocos partidos, una semana con más o menos sorpresas de lo habitual es normal y no dice nada del modelo.

## 3. Resultado

No hubo hipótesis previa. El dashboard es una página de lectura, no un experimento; este notebook solo mide qué tan fiable es lo que muestra:

| Pregunta | Resultado | Lectura |
|---|---|---|
| ¿Elo mejor que la moneda en 2026-27? | Log loss 0.623 [0.555, 0.691] con 107 partidos | Mejor que la moneda, pero el intervalo es ancho y casi la toca |
| ¿Sorpresas esperadas? | 2025-26: 149 observadas vs ~155 esperadas; 2026-27: 15 vs ~14 | Coherente con un Elo calibrado |

**Qué mejorar (no hecho):** incorporar lesiones y descansos (el Elo no los ve); dejar que el usuario elija la fecha; mostrar la calibración acumulada del Elo durante la temporada en la propia página.

## 4. Tus ejercicios

1. **Predice y mide.** Antes de abrir la tabla de la sección 1, escribe si esperas que el Elo gane o pierda contra la moneda en 2026-27 y qué tan seguro estás. Luego compáralo con el intervalo: ¿tu certeza estaba justificada?
2. **Cuánto hace falta.** ¿Cuántos partidos harían falta para que el intervalo del log loss mida ±0.02? (pista: el ancho baja con la raíz del número de partidos).